# Data Preprocessing — Vietnamese GPT-2 Math Word Problems

Input mặc định:

```text
data/raw/train.json
data/raw/valid.json  # optional
```

Output chính:

```text
data/processed/basic_preprocessing/
├── train.jsonl
├── valid_random.jsonl
├── valid_grouped.jsonl
├── valid_public.jsonl       # nếu raw/valid.json tồn tại
├── metrics_pretrain.json
├── preprocessing_report.md
├── variant_config.json
├── drop_log.jsonl
├── transform_log.jsonl
├── sample_diffs.jsonl
└── split_indices/split_indices.json
```


## 1. Imports and global config

In [12]:
from __future__ import annotations

import json
import re
import csv
import math
import random
import hashlib
import statistics
import shutil
import sys
import os
import platform
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

try:
    import pandas as pd
except ImportError:
    pd = None

SEED = 42
random.seed(SEED)

OVERWRITE_OUTPUT = True

# No train split: use separate raw train and raw valid files.
USE_EXTERNAL_VALID = True
CREATE_TRAIN_SPLITS = False

# Input files.
RAW_TRAIN_FILENAME = "train.json"
RAW_VALID_FILENAME = "valid.json"

# Output naming.
TRAIN_OUTPUT_FILENAME = "train.jsonl"
VALID_OUTPUT_FILENAME = "valid.jsonl"

# Disable old split ratios.
VALID_RANDOM_RATIO = 0.0
VALID_GROUPED_RATIO = 0.0
PROCESS_RAW_VALID = True

# Safe default: drop all train rows for unresolved query-level answer conflicts.
# Apply to TRAIN ONLY, not valid.
CONFLICT_POLICY_IF_NO_MANUAL_DECISION = "drop_all"  # allowed: "drop_all", "keep_first"

# V06 policy: keep [asy] if the total sample is short; strip [asy] if the sample is long.
ASY_CONDITIONAL_MAX_TOTAL_CHARS = 1600

MAX_SAMPLE_DIFFS = 500
MAX_LOG_PREVIEW_CHARS = 500

VARIANT_NAME = "v06_keep_asy_if_short"

print("Python:", sys.version)

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]


## 2. Locate data paths

In [13]:
def find_data_dir() -> Path:
    """
    Locate the repository's data directory.

    Expected structures:
    1. Run from data/preprocessing:
       data/raw/train.json
    2. Run from project root:
       data/raw/train.json
    """
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for p in candidates:
        if (p / "raw" / "train.json").exists():
            return p

    for p in candidates:
        if (p / "data" / "raw" / "train.json").exists():
            return p / "data"

    raise FileNotFoundError(
        "Cannot locate data/raw/train.json. "
        "Run this notebook from data/preprocessing, data/, or the project root."
    )

DATA_DIR = find_data_dir()
RAW_DIR = DATA_DIR / "raw"
RAW_TRAIN_PATH = DATA_DIR / "raw" / RAW_TRAIN_FILENAME
RAW_VALID_PATH = DATA_DIR / "raw" / RAW_VALID_FILENAME

# Final clean output: no ablation folders, no multi-version structure.
OUTPUT_DIR = DATA_DIR / "processed" / VARIANT_NAME
TRAIN_OUTPUT_PATH = OUTPUT_DIR / TRAIN_OUTPUT_FILENAME
VALID_OUTPUT_PATH = OUTPUT_DIR / VALID_OUTPUT_FILENAME

SPLIT_DIR = OUTPUT_DIR / "split_indices"
MANUAL_REVIEW_DIR = DATA_DIR / "manual_review"

print("DATA_DIR:", DATA_DIR)
print("RAW_TRAIN_PATH:", RAW_TRAIN_PATH)
print("RAW_VALID_PATH:", RAW_VALID_PATH, "| exists:", RAW_VALID_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_DIR: D:\.Kỳ II năm Ba\DL\BTL\viet-gpt2-math-word-problems\data
RAW_TRAIN_PATH: D:\.Kỳ II năm Ba\DL\BTL\viet-gpt2-math-word-problems\data\raw\train.json
RAW_VALID_PATH: D:\.Kỳ II năm Ba\DL\BTL\viet-gpt2-math-word-problems\data\raw\valid.json | exists: True
OUTPUT_DIR: D:\.Kỳ II năm Ba\DL\BTL\viet-gpt2-math-word-problems\data\processed\v06_keep_asy_if_short


## 3. I/O and utility helpers

In [14]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def read_json_or_jsonl(path: Path) -> Any:
    """
    Read flexible JSON formats:
    1. Standard JSON array/object
    2. JSONL/NDJSON: one valid JSON object per line
    3. Concatenated JSON objects:
       {"a": 1}
       {"b": 2}
       or {"a": 1}{"b": 2}

    Returns:
        list[dict] if multiple records are found
        dict/list if standard JSON document is found
    """
    with path.open("r", encoding="utf-8-sig") as f:
        text = f.read()

    text = text.strip()
    if not text:
        return []

    # Case 1: standard JSON.
    try:
        return json.loads(text)
    except json.JSONDecodeError as normal_json_error:
        pass

    # Case 2 + 3: parse multiple JSON values from one text buffer.
    decoder = json.JSONDecoder()
    records = []
    idx = 0
    n = len(text)

    while idx < n:
        # Skip whitespace between JSON objects.
        while idx < n and text[idx].isspace():
            idx += 1

        if idx >= n:
            break

        try:
            obj, end = decoder.raw_decode(text, idx)
            records.append(obj)
            idx = end
        except json.JSONDecodeError as e:
            context_start = max(0, idx - 300)
            context_end = min(n, idx + 700)
            context = text[context_start:context_end]

            raise ValueError(
                f"Cannot parse JSON object in file: {path}\n"
                f"Parsed records before failure: {len(records)}\n"
                f"Failure index: {idx}\n"
                f"Failure line/column: line {e.lineno}, column {e.colno}\n"
                f"Original error: {e}\n\n"
                f"Context around failure:\n{context}"
            )

    return records

def write_json(obj: Any, path: Path, indent: int = 2) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8", newline="\n") as f:
        json.dump(obj, f, ensure_ascii=False, indent=indent)

def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False, separators=(",", ":")) + "\n")

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def write_text(text: str, path: Path) -> None:
    ensure_dir(path.parent)
    path.write_text(text, encoding="utf-8", newline="\n")

def md5_text(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def stable_hash(obj: Any) -> str:
    s = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def dataset_fingerprint(records: List[Dict[str, Any]]) -> str:
    payload = [
        {
            "id": r.get("id"),
            "query": r.get("query"),
            "response": r.get("response"),
            "final_answer": r.get("final_answer"),
            "type": r.get("type"),
        }
        for r in records
    ]
    return stable_hash(payload)

def preview(text: Any, n: int = MAX_LOG_PREVIEW_CHARS) -> str:
    if text is None:
        return ""
    s = str(text)
    return s[:n] + ("..." if len(s) > n else "")

def percentile(values: List[int | float], p: float) -> Optional[float]:
    if not values:
        return None
    values = sorted(values)
    if len(values) == 1:
        return values[0]
    k = (len(values) - 1) * p
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return values[int(k)]
    return values[f] * (c - k) + values[c] * (k - f)

## 4. Load raw data and build base records

In [15]:
def get_source_group(type_value: str) -> str:
    t = str(type_value or "")
    if "_" in t:
        return t.split("_", 1)[0]
    return t or "UNKNOWN"

def get_aug_type(type_value: str) -> str:
    t = str(type_value or "")
    if "_" in t:
        return t.split("_", 1)[1]
    return "UNKNOWN"

def normalize_for_hash(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()

def has_latex(text: str) -> bool:
    s = str(text or "")
    return bool(re.search(r"(\\frac|\\sqrt|\\boxed|\\left|\\right|\\cdot|\\times|\\angle|\\sum|\\int|[$^_{}])", s))

def has_asy(text: str) -> bool:
    return bool(re.search(r"\[asy\].*?\[/asy\]", str(text or ""), flags=re.IGNORECASE | re.DOTALL))

def has_decimal_comma(text: str) -> bool:
    return bool(re.search(r"(?<!\d)\d+,\d{1,2}(?!\d)", str(text or "")))

def make_base_record(raw: Dict[str, Any], idx: int, dataset_name: str) -> Dict[str, Any]:
    query_raw = str(raw.get("query_vi", "") or "")
    response_raw = str(raw.get("response_vi", "") or "")
    type_value = str(raw.get("type", "UNKNOWN") or "UNKNOWN")
    original_question = str(raw.get("original_question_vi", "") or query_raw)

    return {
        "id": f"{dataset_name}_{idx:06d}",
        "raw_index": idx,
        "dataset_name": dataset_name,

        "query_raw": query_raw,
        "response_raw": response_raw,
        "original_question_vi": original_question,

        "type": type_value,
        "source_group": get_source_group(type_value),
        "aug_type": get_aug_type(type_value),

        "original_question_hash": md5_text(normalize_for_hash(original_question)),
        "query_raw_hash": md5_text(normalize_for_hash(query_raw)),
        "query_response_raw_hash": md5_text(normalize_for_hash(query_raw) + "\n" + normalize_for_hash(response_raw)),

        "has_latex_original": has_latex(query_raw) or has_latex(response_raw),
        "has_asy_original": has_asy(query_raw) or has_asy(response_raw),
        "has_decimal_comma_original": has_decimal_comma(query_raw) or has_decimal_comma(response_raw),
    }

raw_train = read_json_or_jsonl(RAW_TRAIN_PATH)
if not isinstance(raw_train, list):
    raise ValueError("Raw train file must contain a list of records or JSONL records.")

raw_valid = read_json_or_jsonl(RAW_VALID_PATH)
if not isinstance(raw_valid, list):
    raise ValueError("Raw valid file must contain a list of records or JSONL records.")

print("Raw train records:", len(raw_train))
print("Raw valid records:", len(raw_valid))
print("First train keys:", list(raw_train[0].keys()) if raw_train else [])
print("First valid keys:", list(raw_valid[0].keys()) if raw_valid else [])

REQUIRED_FIELDS = ["query_vi", "response_vi", "type"]
missing = []
for i, r in enumerate(raw_train[:100]):
    for f in REQUIRED_FIELDS:
        if f not in r:
            missing.append((i, f))
if missing:
    raise ValueError(f"Missing required fields in train sample: {missing[:20]}")

base_train = [make_base_record(r, i, "train") for i, r in enumerate(raw_train)]
base_valid = [make_base_record(r, i, "valid") for i, r in enumerate(raw_valid)]

print("Base train:", len(base_train))
print("Base valid:", len(base_valid))
print("Type distribution:", dict(Counter(r["type"] for r in base_train).most_common(20)))

Raw train records: 95400
Raw valid records: 1000
First train keys: ['original_question_vi', 'original_question_en', 'query_vi', 'query_en', 'response_vi', 'response_en', 'type']
First valid keys: ['original_question_vi', 'original_question_en', 'query_vi', 'query_en', 'response_vi', 'response_en', 'type']
Base train: 95400
Base valid: 1000
Type distribution: {'GSM_Rephrased': 20028, 'GSM_AnsAug': 18745, 'MATH_AnsAug': 16999, 'MATH_Rephrased': 12477, 'GSM_FOBAR': 10023, 'GSM_SV': 9869, 'MATH_FOBAR': 3668, 'MATH_SV': 3591}


## 5. Create train / valid_random / valid_grouped split

In [17]:
split_payload = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "use_external_valid": USE_EXTERNAL_VALID,
    "create_train_splits": CREATE_TRAIN_SPLITS,
    "input_counts": {
        "raw_train": len(base_train),
        "raw_valid": len(base_valid),
    },
    "output_splits": {
        "train": len(base_train),
        "valid": len(base_valid),
    },
    "input_files": {
        "raw_train": str(RAW_TRAIN_PATH),
        "raw_valid": str(RAW_VALID_PATH),
    },
}

base_splits = {
    "train": base_train,
    "valid": base_valid,
}

print(json.dumps(split_payload, ensure_ascii=False, indent=2))

for split_name, records in base_splits.items():
    print(split_name, len(records))

{
  "created_at": "2026-05-20T20:34:17",
  "seed": 42,
  "use_external_valid": true,
  "create_train_splits": false,
  "input_counts": {
    "raw_train": 95400,
    "raw_valid": 1000
  },
  "output_splits": {
    "train": 95400,
    "valid": 1000
  },
  "input_files": {
    "raw_train": "D:\\.Kỳ II năm Ba\\DL\\BTL\\viet-gpt2-math-word-problems\\data\\raw\\train.json",
    "raw_valid": "D:\\.Kỳ II năm Ba\\DL\\BTL\\viet-gpt2-math-word-problems\\data\\raw\\valid.json"
  }
}
train 95400
valid 1000


## 6. Text cleaning, answer extraction, and normalization helpers

In [18]:
def fix_artifacts(text: str) -> Tuple[str, List[Dict[str, Any]]]:
    logs = []
    original = text
    text = str(text or "")

    replacements = [
        (r"\đóng hộp{", r"\boxed{", "fix_artifact_dong_hop"),
        ("\u200b", "", "remove_zero_width_space"),
        ("\ufeff", "", "remove_bom"),
    ]
    for old, new, name in replacements:
        if old in text:
            text = text.replace(old, new)
            logs.append({"transform_type": name})

    if text != original:
        logs.append({"transform_type": "fix_artifacts_any", "before_length": len(original), "after_length": len(text)})
    return text.strip(), logs

def clean_text_basic(text: str) -> Tuple[str, List[Dict[str, Any]]]:
    logs = []
    original = str(text or "")
    text = original

    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    text2 = re.sub(r"(Giá trị của biến [^\n?]+\?)\s*\1", r"\1", text)
    if text2 != text:
        logs.append({"transform_type": "clean_repeated_variable_sentence"})
    text = text2

    text2 = re.sub(
        r"\n(?:The answer is[:\s]+[^\n]+\.?\s*)+$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    if text2 != text:
        logs.append({"transform_type": "remove_english_leakage_tail"})
    text = text2

    text = text.strip()
    if text != original:
        logs.append({"transform_type": "clean_text_basic", "before_length": len(original), "after_length": len(text)})
    return text, logs

def strip_asy_blocks(text: str) -> Tuple[str, List[Dict[str, Any]]]:
    original = str(text or "")
    pattern = re.compile(r"\[asy\].*?\[/asy\]", flags=re.IGNORECASE | re.DOTALL)
    matches = list(pattern.finditer(original))
    text = pattern.sub("", original)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text).strip()
    logs = []
    if matches:
        logs.append({
            "transform_type": "strip_asy",
            "asy_block_count": len(matches),
            "before_length": len(original),
            "after_length": len(text),
        })
    return text, logs

def normalize_decimal_format(text: str) -> Tuple[str, List[Dict[str, Any]]]:
    """
    Conservative decimal-comma normalization.

    Rules:
    - Convert clear decimal comma patterns: 2,5 → 2.5; -3,14 → -3.14.
    - Convert European-style: 1.200,5 → 1200.5.
    - Do not attempt aggressive normalization inside complex LaTeX.
    """
    original = str(text or "")
    text = original

    text = re.sub(
        r"\\boxed\{(-?\d+),(\d+)\}",
        lambda m: f"\\boxed{{{m.group(1)}.{m.group(2)}}}",
        text,
    )

    text = re.sub(
        r"(?<!\w)(-?\d{1,3}(?:\.\d{3})+),(\d{1,2})(?!\d)",
        lambda m: f"{m.group(1).replace('.', '')}.{m.group(2)}",
        text,
    )

    text = re.sub(
        r"(?<![A-Za-zÀ-ỹ\\])(-?\d+),(\d{1,2})(?!\d)(?![A-Za-zÀ-ỹ}])",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )

    logs = []
    if text != original:
        logs.append({
            "transform_type": "normalize_decimal",
            "before_length": len(original),
            "after_length": len(text),
        })
    return text, logs

def extract_balanced_command_content(text: str, command: str = "boxed") -> List[str]:
    """
    Extract contents of LaTeX commands like \boxed{...} with balanced braces.
    Returns all contents found.
    """
    s = str(text or "")
    marker = "\\" + command + "{"
    results = []
    start = 0

    while True:
        i = s.find(marker, start)
        if i == -1:
            break

        content_start = i + len(marker)
        depth = 1
        j = content_start
        while j < len(s):
            if s[j] == "{":
                depth += 1
            elif s[j] == "}":
                depth -= 1
                if depth == 0:
                    results.append(s[content_start:j])
                    start = j + 1
                    break
            j += 1
        else:
            break

    return results

def normalize_answer(answer: Optional[str]) -> str:
    if answer is None:
        return ""

    a = str(answer).strip()
    a = re.sub(r"\s+", " ", a)
    a = a.strip(" .。")

    # Bỏ prefix thừa kiểu "Giá trị 25"
    a = re.sub(r"^(?:Giá trị|giá trị)\s+", "", a).strip()

    # Normalize LaTeX display fraction
    a = a.replace(r"\dfrac", r"\frac")
    a = a.replace(r"\tfrac", r"\frac")

    # Normalize pure decimal comma:
    # "251,60" -> "251.60"
    # "4,5" -> "4.5"
    # Không đổi tuple/coordinate như "(-1.1,1)"
    if re.fullmatch(r"-?\d+,\d+", a):
        a = a.replace(",", ".")

    # Bỏ unit đứng sau số plain
    # "150 km" -> "150"
    # Không động vào LaTeX/symbolic
    if not re.search(r"[\\{}^_]", a):
        a = re.sub(r"^(-?[\d]+(?:\.\d+)?(?:/\d+)?)\s+[A-Za-zÀ-ỹ%°]+.*$", r"\1", a)

    return a.strip()

def clean_extracted_answer_candidate(ans: str) -> str:
    ans = str(ans or "").strip()

    # Nếu candidate vẫn chứa anchor khác, lấy phần sau anchor cuối cùng.
    # Ví dụ: "$\\boxed{3}$.Câu trả lời là: 3" -> "3"
    anchor_inner = list(re.finditer(
        r"(?:Đáp án là|Câu trả lời là|####)[:\s]*",
        ans,
        flags=re.IGNORECASE,
    ))
    if anchor_inner:
        ans = ans[anchor_inner[-1].end():].strip()

    # Nếu có boxed trong candidate, ưu tiên nội dung boxed cuối.
    boxed = extract_balanced_command_content(ans, "boxed")
    if boxed:
        ans = boxed[-1].strip()

    # Cắt các cụm diễn giải sau đáp án nếu còn sót.
    # Ví dụ: "25. Đáp án là: 25" đã xử lý ở trên;
    # "6 nên chúng ta..." thì fallback sau sẽ bắt số cuối tốt hơn.
    ans = ans.strip(" .。")

    return normalize_answer(ans)

def is_bad_final_answer_candidate(ans: str) -> bool:
    s = str(ans or "").strip()

    if not s:
        return True

    # Quá dài thì gần như chắc là bắt nhầm lời giải.
    if len(s) > 80:
        return True

    # Có cụm diễn giải thì không phải answer compact.
    bad_markers = [
        "nên chúng ta",
        "vì vậy",
        "do đó",
        "để giải",
        "thiết lập phương trình",
        "chúng ta có",
        "ta có",
    ]

    lower = s.lower()
    if any(marker in lower for marker in bad_markers):
        return True

    return False


def extract_compact_answer_from_text(text: str) -> Optional[str]:
    s = str(text or "")

    # Nếu có boxed, lấy boxed cuối.
    boxed = extract_balanced_command_content(s, "boxed")
    if boxed:
        return boxed[-1].strip()

    # Nếu có dạng "Câu trả lời là: X" bên trong, lấy X.
    inner = list(re.finditer(
        r"(?:Đáp án là|Câu trả lời là|####)[:\s]*([^\n\.。]+)",
        s,
        flags=re.IGNORECASE,
    ))
    if inner:
        return inner[-1].group(1).strip()

    # Lấy biểu thức compact cuối cùng.
    pattern = re.compile(
        r"(\\frac\{[^{}]+\}\{[^{}]+\}|"
        r"\\sqrt\{[^{}]+\}|"
        r"-?\d+(?:\.\d+|,\d+)?(?:_\d+)?|"
        r"[A-Za-z]\s*=\s*-?\d+(?:\.\d+|,\d+)?)"
    )

    nums = pattern.findall(s)
    if nums:
        return nums[-1].strip()

    return None

def extract_final_answer(response: str) -> Tuple[Optional[str], str]:
    """
    Extract final answer robustly.

    Principle:
    - Do not prioritize anchor type.
    - Prioritize the last answer-like anchor appearing near the end.
    - Avoid taking a whole explanation sentence as final_answer.
    """
    text = str(response or "")
    tail = text[-2000:]

    anchor_regex = re.compile(
        r"(Đáp án là|Câu trả lời là|####)[:\s]*([^\n]*)",
        flags=re.IGNORECASE,
    )

    matches = list(anchor_regex.finditer(tail))

    if matches:
        # Lấy anchor cuối cùng trong tail, bất kể là "Đáp án là" hay "Câu trả lời là".
        m = matches[-1]
        raw_ans = m.group(2).strip()
        ans = clean_extracted_answer_candidate(raw_ans)

        # Nếu ans vẫn quá dài / giống câu giải thích, thử fallback từ chính raw_ans.
        if is_bad_final_answer_candidate(ans):
            fallback = extract_compact_answer_from_text(raw_ans)
            if fallback:
                return normalize_answer(fallback), "anchor_last_fallback_compact"

        return ans, "anchor_last"

    # GSM style fallback
    matches = list(re.finditer(r"####\s*([^\n]+)", tail))
    if matches:
        ans = clean_extracted_answer_candidate(matches[-1].group(1))
        return ans, "gsm_hash"

    # LaTeX boxed fallback
    boxed_contents = extract_balanced_command_content(tail, "boxed")
    if boxed_contents:
        return normalize_answer(boxed_contents[-1]), "boxed"

    # Last compact expression fallback
    fallback = extract_compact_answer_from_text(tail)
    if fallback:
        return normalize_answer(fallback), "fallback_compact"

    return None, "failed"

def remove_existing_final_answer_tail(response: str) -> str:
    text = str(response or "").rstrip()

    # Nếu cuối response có cụm answer anchor, xóa từ anchor cuối cùng tới hết.
    # Lặp vài lần để xóa case:
    # "... Đáp án là: $\\boxed{3}$.Câu trả lời là: 3"
    for _ in range(5):
        matches = list(re.finditer(
            r"(Đáp án là|Câu trả lời là|####)[:\s]*",
            text,
            flags=re.IGNORECASE,
        ))

        if not matches:
            break

        last = matches[-1]
        tail_after_anchor = text[last.start():]

        # Chỉ xóa nếu anchor nằm gần cuối, tránh xóa nhầm anchor giữa lời giải quá xa.
        if len(tail_after_anchor) <= 500:
            text = text[:last.start()].rstrip(" .。:\n\t")
        else:
            break

    return text.rstrip()

def rebuild_response_with_anchor(response: str, answer: str) -> str:
    base = remove_existing_final_answer_tail(response)
    return base.rstrip() + f"\nĐáp án là: {answer}"

def canonical_answer_for_conflict(answer: str) -> str:
    s = normalize_answer(answer)
    s = str(s or "").strip()

    # Xóa whitespace toàn bộ để normalize LaTeX/simple expression
    s = re.sub(r"\s+", "", s)

    # Normalize LaTeX fraction variants
    s = s.replace(r"\dfrac", r"\frac")
    s = s.replace(r"\tfrac", r"\frac")
    s = re.sub(
        r"\\frac\s*\{\s*([^{}]+?)\s*\}\s*\{\s*([^{}]+?)\s*\}",
        r"\\frac{\1}{\2}",
        s,
    )
    s = re.sub(
        r"\\sqrt\s*\{\s*([^{}]+?)\s*\}",
        r"\\sqrt{\1}",
        s,
    )

    # Remove simple translated prefix
    s = re.sub(r"^(?:Giátrị|giátrị)", "", s)

    # Remove trailing text unit in LaTeX
    s = re.sub(r"\\text\{[^{}]*\}$", "", s)

    # Normalize pure decimal comma only
    if re.fullmatch(r"-?\d+,\d+", s):
        s = s.replace(",", ".")

    # Strip parentheses only for single numeric value, not coordinate tuple
    if re.fullmatch(r"\(-?\d+(?:\.\d+)?\)", s):
        s = s[1:-1]

    return s

## 7. V06 config and per-record preprocessing

In [19]:
V06_CONFIG: Dict[str, Any] = {
    "fix_artifacts": True,
    "clean_text": True,
    "normalize_anchor": True,
    "dedup": True,
    "resolve_conflicts": True,
    "normalize_decimal": True,
    "asy_policy": "conditional",
    "asy_conditional_max_total_chars": ASY_CONDITIONAL_MAX_TOTAL_CHARS,
    "description": (
        "Final selected preprocessing policy: fix LaTeX artifacts, clean text, "
        "normalize decimals, normalize final answer anchor, exact dedup, "
        "query-level conflict dropping, and keep [asy] only when the sample is short."
    ),
}

print(json.dumps(V06_CONFIG, ensure_ascii=False, indent=2))


def apply_asy_policy(
    query: str,
    response: str,
    config: Dict[str, Any],
) -> Tuple[str, str, List[Dict[str, Any]]]:
    logs = []
    policy = config.get("asy_policy", "keep")

    if policy == "keep":
        return query, response, logs

    if policy == "strip":
        q2, qlogs = strip_asy_blocks(query)
        r2, rlogs = strip_asy_blocks(response)
        logs.extend([{"field": "query", **x} for x in qlogs])
        logs.extend([{"field": "response", **x} for x in rlogs])
        return q2, r2, logs

    if policy == "conditional":
        max_chars = int(config.get("asy_conditional_max_total_chars", ASY_CONDITIONAL_MAX_TOTAL_CHARS))
        has_any_asy = has_asy(query) or has_asy(response)
        total_chars = len(query) + len(response)
        if has_any_asy and total_chars > max_chars:
            q2, qlogs = strip_asy_blocks(query)
            r2, rlogs = strip_asy_blocks(response)
            logs.append({
                "transform_type": "conditional_strip_asy_triggered",
                "total_chars_before": total_chars,
                "max_total_chars": max_chars,
            })
            logs.extend([{"field": "query", **x} for x in qlogs])
            logs.extend([{"field": "response", **x} for x in rlogs])
            return q2, r2, logs
        return query, response, logs

    raise ValueError(f"Unknown asy_policy: {policy}")

def preprocess_one_record(
    base: Dict[str, Any],
    config: Dict[str, Any],
    variant_name: str,
    split_name: str,
) -> Tuple[Dict[str, Any], List[Dict[str, Any]], List[Dict[str, Any]]]:
    query = str(base.get("query_raw", "") or "").strip()
    response = str(base.get("response_raw", "") or "").strip()

    transform_logs = []
    sample_diffs = []

    original_query, original_response = query, response

    def add_logs(logs, field):
        for log in logs:
            transform_logs.append({
                "variant": variant_name,
                "split": split_name,
                "id": base["id"],
                "raw_index": base["raw_index"],
                "type": base["type"],
                "field": field,
                **log,
            })

    if config.get("fix_artifacts"):
        query, logs = fix_artifacts(query)
        add_logs(logs, "query")
        response, logs = fix_artifacts(response)
        add_logs(logs, "response")

    query, response, asy_logs = apply_asy_policy(query, response, config)
    for log in asy_logs:
        transform_logs.append({
            "variant": variant_name,
            "split": split_name,
            "id": base["id"],
            "raw_index": base["raw_index"],
            "type": base["type"],
            **log,
        })

    if config.get("clean_text"):
        query, logs = clean_text_basic(query)
        add_logs(logs, "query")
        response, logs = clean_text_basic(response)
        add_logs(logs, "response")

    if config.get("normalize_decimal"):
        q_before, r_before = query, response
        query, logs = normalize_decimal_format(query)
        add_logs(logs, "query")
        response, logs = normalize_decimal_format(response)
        add_logs(logs, "response")
        if (query != q_before or response != r_before):
            sample_diffs.append({
                "variant": variant_name,
                "split": split_name,
                "id": base["id"],
                "raw_index": base["raw_index"],
                "type": base["type"],
                "diff_type": "normalize_decimal",
                "before_query": preview(q_before),
                "after_query": preview(query),
                "before_response": preview(r_before),
                "after_response": preview(response),
            })

    final_answer, answer_method = extract_final_answer(response)
    answer_extract_failed = final_answer is None or final_answer == ""

    was_anchor_rebuilt = False
    if config.get("normalize_anchor") and not answer_extract_failed:
        r_before = response
        response = rebuild_response_with_anchor(response, final_answer)
        was_anchor_rebuilt = response != r_before
        if was_anchor_rebuilt:
            transform_logs.append({
                "variant": variant_name,
                "split": split_name,
                "id": base["id"],
                "raw_index": base["raw_index"],
                "type": base["type"],
                "field": "response",
                "transform_type": "normalize_anchor",
                "before_length": len(r_before),
                "after_length": len(response),
            })

    record = {
        "id": base["id"],
        "raw_index": base["raw_index"],
        "dataset_name": base["dataset_name"],
        "split": split_name,

        "query": query,
        "response": response,
        "final_answer": final_answer or "",
        "answer_conflict_key": canonical_answer_for_conflict(final_answer or ""),

        "type": base["type"],
        "source_group": base["source_group"],
        "aug_type": base["aug_type"],

        "original_question_hash": base["original_question_hash"],
        "query_hash": md5_text(normalize_for_hash(query)),
        "query_response_hash": md5_text(normalize_for_hash(query) + "\n" + normalize_for_hash(response)),

        "has_latex_original": base["has_latex_original"],
        "has_latex_after": has_latex(query) or has_latex(response),

        "has_asy_original": base["has_asy_original"],
        "has_asy_after": has_asy(query) or has_asy(response),
        "was_asy_stripped": base["has_asy_original"] and not (has_asy(query) or has_asy(response)),

        "has_decimal_comma_original": base["has_decimal_comma_original"],
        "has_decimal_comma_after": has_decimal_comma(query) or has_decimal_comma(response),
        "was_decimal_normalized": bool(config.get("normalize_decimal", False)) and (
            base["has_decimal_comma_original"] != (has_decimal_comma(query) or has_decimal_comma(response)) or
            query != original_query or response != original_response
        ),

        "was_artifact_fixed": bool(config.get("fix_artifacts", False)) and (query != original_query or response != original_response),
        "was_anchor_rebuilt": was_anchor_rebuilt,

        "answer_extract_method": answer_method,
        "answer_extract_failed": bool(answer_extract_failed),

        "preprocess_variant": variant_name,
    }

    if query != original_query or response != original_response:
        if len(sample_diffs) < 1:
            sample_diffs.append({
                "variant": variant_name,
                "split": split_name,
                "id": base["id"],
                "raw_index": base["raw_index"],
                "type": base["type"],
                "diff_type": "general_transform",
                "before_query": preview(original_query),
                "after_query": preview(query),
                "before_response": preview(original_response),
                "after_response": preview(response),
            })

    return record, transform_logs, sample_diffs

{
  "fix_artifacts": true,
  "clean_text": true,
  "normalize_anchor": true,
  "dedup": true,
  "resolve_conflicts": true,
  "normalize_decimal": true,
  "asy_policy": "conditional",
  "asy_conditional_max_total_chars": 1600,
  "description": "Final selected preprocessing policy: fix LaTeX artifacts, clean text, normalize decimals, normalize final answer anchor, exact dedup, query-level conflict dropping, and keep [asy] only when the sample is short."
}


## 8. Train-only deduplication and conflict handling

In [20]:
def export_conflicts_for_review(conflict_groups: Dict[str, List[Dict[str, Any]]], out_csv: Path) -> None:
    ensure_dir(out_csv.parent)
    rows = []
    for query_hash, group in conflict_groups.items():
        answers = Counter(r.get("final_answer", "") for r in group)
        rows.append({
            "query_hash": query_hash,
            "query_preview": preview(group[0].get("query", ""), 300),
            "type_values": "|".join(sorted(set(r.get("type", "") for r in group))),
            "num_rows": len(group),
            "num_unique_answers": len(answers),
            "answer_counts": json.dumps(dict(answers), ensure_ascii=False),
            "sample_ids": "|".join(r.get("id", "") for r in group[:10]),
            "manual_decision": "",
            "notes": "",
        })

    if not rows:
        return

    if pd is not None:
        pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")
    else:
        with out_csv.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)

def load_conflict_decisions(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}
    data = read_json_or_jsonl(path)
    if isinstance(data, dict):
        return data
    if isinstance(data, list):
        return {str(x.get("query_hash") or x.get("query")): x for x in data}
    return {}

def apply_dedup_and_conflicts(
    records: List[Dict[str, Any]],
    variant_name: str,
    config: Dict[str, Any],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    drop_logs = []
    result = records

    non_empty = []
    for r in result:
        if not str(r.get("query", "")).strip() or not str(r.get("response", "")).strip():
            drop_logs.append({
                "variant": variant_name,
                "split": "train",
                "id": r.get("id"),
                "raw_index": r.get("raw_index"),
                "drop_reason": "empty_after_clean",
                "type": r.get("type"),
                "query_preview": preview(r.get("query")),
                "response_tail": preview(str(r.get("response", ""))[-300:]),
                "final_answer": r.get("final_answer", ""),
            })
        else:
            non_empty.append(r)
    result = non_empty

    if config.get("normalize_anchor"):
        kept = []
        for r in result:
            bad_answer = is_bad_final_answer_candidate(r.get("final_answer", ""))

            if r.get("answer_extract_failed") or bad_answer:
                drop_logs.append({
                    "variant": variant_name,
                    "split": "train",
                    "id": r.get("id"),
                    "raw_index": r.get("raw_index"),
                    "drop_reason": "extract_failed"
                        if r.get("answer_extract_failed")
                        else "bad_final_answer_candidate",
                    "type": r.get("type"),
                    "query_preview": preview(r.get("query")),
                    "response_tail": preview(str(r.get("response", ""))[-300:]),
                    "final_answer": r.get("final_answer", ""),
                })
            else:
                kept.append(r)

        result = kept

    if config.get("dedup"):
        seen = set()
        deduped = []
        for r in result:
            key = r.get("query_response_hash")
            if key in seen:
                drop_logs.append({
                    "variant": variant_name,
                    "split": "train",
                    "id": r.get("id"),
                    "raw_index": r.get("raw_index"),
                    "drop_reason": "exact_duplicate",
                    "type": r.get("type"),
                    "query_preview": preview(r.get("query")),
                    "response_tail": preview(str(r.get("response", ""))[-300:]),
                    "final_answer": r.get("final_answer", ""),
                })
            else:
                seen.add(key)
                deduped.append(r)
        result = deduped

    if config.get("resolve_conflicts"):
        query_groups = defaultdict(list)
        for r in result:
            query_groups[r["query_hash"]].append(r)

        conflict_groups = {
            qh: group
            for qh, group in query_groups.items()
            if len(set(g.get("answer_conflict_key", "") for g in group)) > 1
        }

        if conflict_groups:
            export_conflicts_for_review(
                conflict_groups,
                MANUAL_REVIEW_DIR / "conflicts_to_review.csv",
            )

        decisions = load_conflict_decisions(MANUAL_REVIEW_DIR / "conflicts_decisions.json")
        kept = []

        for r in result:
            qh = r["query_hash"]
            if qh not in conflict_groups:
                kept.append(r)
                continue

            decision_obj = decisions.get(qh, None)
            decision = None
            if isinstance(decision_obj, dict):
                decision = decision_obj.get("manual_decision") or decision_obj.get("answer")
            elif isinstance(decision_obj, str):
                decision = decision_obj

            if decision:
                if decision == "drop_all":
                    reason = "conflict_dropped_manual_drop_all"
                    should_keep = False
                elif str(r.get("answer_conflict_key", "")) == canonical_answer_for_conflict(str(decision)):
                    should_keep = True
                    reason = ""
                else:
                    should_keep = False
                    reason = "conflict_dropped_manual_answer_mismatch"
            else:
                if CONFLICT_POLICY_IF_NO_MANUAL_DECISION == "drop_all":
                    should_keep = False
                    reason = "conflict_dropped_no_manual_decision"
                elif CONFLICT_POLICY_IF_NO_MANUAL_DECISION == "keep_first":
                    first_answer = conflict_groups[qh][0].get("final_answer", "")
                    should_keep = r.get("final_answer", "") == first_answer
                    reason = "" if should_keep else "conflict_dropped_keep_first"
                else:
                    raise ValueError(f"Unknown conflict policy: {CONFLICT_POLICY_IF_NO_MANUAL_DECISION}")

            if should_keep:
                kept.append(r)
            else:
                drop_logs.append({
                    "variant": variant_name,
                    "split": "train",
                    "id": r.get("id"),
                    "raw_index": r.get("raw_index"),
                    "drop_reason": reason,
                    "type": r.get("type"),
                    "query_preview": preview(r.get("query")),
                    "response_tail": preview(str(r.get("response", ""))[-300:]),
                    "final_answer": r.get("final_answer", ""),
                })

        result = kept

    return result, drop_logs


## 9. Metrics and report helpers

In [21]:
def count_exact_duplicates(records: List[Dict[str, Any]]) -> int:
    keys = [r.get("query_response_hash") for r in records]
    return len(keys) - len(set(keys))

def count_conflict_queries(records: List[Dict[str, Any]]) -> Tuple[int, int]:
    q_to_answers = defaultdict(list)

    for r in records:
        q = r.get("query_hash")
        a = r.get("answer_conflict_key", r.get("final_answer", ""))
        q_to_answers[q].append(a)

    conflict_q = 0
    conflict_rows = 0

    for _, answers in q_to_answers.items():
        if len(set(answers)) > 1:
            conflict_q += 1
            conflict_rows += len(answers)

    return conflict_q, conflict_rows

def distribution(records: List[Dict[str, Any]], key: str) -> Dict[str, int]:
    return dict(Counter(str(r.get(key, "UNKNOWN")) for r in records))

def count_answer_anchors(text: str) -> int:
    """Count strict answer anchors with a colon to avoid false positives in prose."""
    return len(re.findall(
        r"(?:Đáp án là|Câu trả lời là|####)\s*[:：]",
        str(text or ""),
        flags=re.IGNORECASE,
    ))

def is_pure_decimal_comma_answer(answer: str) -> bool:
    return bool(re.fullmatch(r"-?\d+,\d+", str(answer or "").strip()))

def split_quality_metrics(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    total = len(records)
    queries = [str(r.get("query", "")) for r in records]
    responses = [str(r.get("response", "")) for r in records]
    total_chars = [len(q) + len(resp) for q, resp in zip(queries, responses)]
    query_chars = [len(q) for q in queries]
    response_chars = [len(resp) for resp in responses]

    answer_extract_failed = sum(1 for r in records if r.get("answer_extract_failed"))
    anchor_has = sum(1 for r in records if "Đáp án là:" in str(r.get("response", "")))
    anchor_last = sum(
        1 for r in records
        if str(r.get("response", "")).rstrip().endswith("Đáp án là: " + str(r.get("final_answer", "")).strip())
        and str(r.get("final_answer", "")).strip() != ""
    )

    old_anchor_remaining = sum(
        1 for r in records
        if "Câu trả lời là:" in str(r.get("response", ""))
    )

    final_answer_empty = sum(1 for r in records if not str(r.get("final_answer", "")).strip())

    duplicate_anchor_count = sum(
        1 for r in records
        if count_answer_anchors(r.get("response", "")) > 1
    )

    pure_decimal_comma_answer_count = sum(
        1 for r in records
        if is_pure_decimal_comma_answer(r.get("final_answer", ""))
    )

    return {
        "count": total,
        "empty_query_count": sum(1 for q in queries if not q.strip()),
        "empty_response_count": sum(1 for resp in responses if not resp.strip()),
        "answer_extract_failed_count": answer_extract_failed,
        "answer_extract_success_rate": None if total == 0 else 1 - answer_extract_failed / total,
        "anchor_has_rate": None if total == 0 else anchor_has / total,
        "anchor_last_line_rate": None if total == 0 else anchor_last / total,
        "old_anchor_remaining_count": old_anchor_remaining,
        "final_answer_empty_count": final_answer_empty,
        "exact_duplicate_remaining": count_exact_duplicates(records),
        "conflict_query_remaining": count_conflict_queries(records)[0],
        "conflict_row_remaining": count_conflict_queries(records)[1],
        "latex_count_after": sum(1 for r in records if r.get("has_latex_after")),
        "latex_count_original": sum(1 for r in records if r.get("has_latex_original")),
        "latex_preservation_ratio": (
            None
            if sum(1 for r in records if r.get("has_latex_original")) == 0
            else sum(1 for r in records if r.get("has_latex_after")) / sum(1 for r in records if r.get("has_latex_original"))
        ),
        "asy_count_original": sum(1 for r in records if r.get("has_asy_original")),
        "asy_count_after": sum(1 for r in records if r.get("has_asy_after")),
        "decimal_comma_answer_remaining_count": sum(1 for r in records if has_decimal_comma(r.get("final_answer", ""))),
        "decimal_comma_query_remaining_count": sum(1 for r in records if has_decimal_comma(r.get("query", ""))),
        "decimal_comma_response_remaining_count": sum(1 for r in records if has_decimal_comma(r.get("response", ""))),
        "query_chars": {
            "p50": percentile(query_chars, 0.50),
            "p90": percentile(query_chars, 0.90),
            "p95": percentile(query_chars, 0.95),
            "p99": percentile(query_chars, 0.99),
            "max": max(query_chars) if query_chars else None,
        },
        "response_chars": {
            "p50": percentile(response_chars, 0.50),
            "p90": percentile(response_chars, 0.90),
            "p95": percentile(response_chars, 0.95),
            "p99": percentile(response_chars, 0.99),
            "max": max(response_chars) if response_chars else None,
        },
        "total_chars": {
            "p50": percentile(total_chars, 0.50),
            "p90": percentile(total_chars, 0.90),
            "p95": percentile(total_chars, 0.95),
            "p99": percentile(total_chars, 0.99),
            "max": max(total_chars) if total_chars else None,
        },
        "type_distribution": distribution(records, "type"),
        "source_group_distribution": distribution(records, "source_group"),
        "aug_type_distribution": distribution(records, "aug_type"),
        "fingerprint": dataset_fingerprint(records),
        "duplicate_anchor_count": duplicate_anchor_count,
        "pure_decimal_comma_answer_count": pure_decimal_comma_answer_count,

        "duplicate_anchor_count": sum(
            1 for r in records
            if count_answer_anchors(r.get("response", "")) > 1
        ),
        "bad_final_answer_candidate_count": sum(
            1 for r in records
            if is_bad_final_answer_candidate(r.get("final_answer", ""))
        ),
        "pure_decimal_comma_answer_count": sum(
            1 for r in records
            if re.fullmatch(r"-?\d+,\d+", str(r.get("final_answer", "")).strip())
        ),
    }

def leakage_metrics(train_records, valid_records) -> Dict[str, int]:
    train_groups = set(r.get("original_question_hash") for r in train_records)
    valid_groups = set(r.get("original_question_hash") for r in valid_records)
    train_queries = set(r.get("query_hash") for r in train_records)
    valid_queries = set(r.get("query_hash") for r in valid_records)
    train_qr = set(r.get("query_response_hash") for r in train_records)
    valid_qr = set(r.get("query_response_hash") for r in valid_records)

    return {
        "group_overlap_count": len(train_groups & valid_groups),
        "query_overlap_count": len(train_queries & valid_queries),
        "query_response_overlap_count": len(train_qr & valid_qr),
    }

def build_metrics_pretrain(
    variant_name: str,
    config: Dict[str, Any],
    split_outputs: Dict[str, List[Dict[str, Any]]],
    drop_logs: List[Dict[str, Any]],
    transform_logs: List[Dict[str, Any]],
) -> Dict[str, Any]:
    train_records = split_outputs.get("train", [])
    valid_records = split_outputs.get("valid", [])

    drop_summary = Counter(x.get("drop_reason", "unknown") for x in drop_logs)
    transform_summary = Counter(x.get("transform_type", "unknown") for x in transform_logs)

    metrics = {
        "variant": variant_name,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "seed": SEED,
        "config": config,
        "counts": {
            "raw_train_count": len(base_train),
            "raw_valid_count": len(base_valid),
            "train_output_count": len(train_records),
            "valid_output_count": len(valid_records),
        },
        "drop_summary": dict(drop_summary),
        "transform_summary": dict(transform_summary),
        "splits": {
            "train": split_quality_metrics(train_records),
            "valid": split_quality_metrics(valid_records),
        },
        "leakage_check": {
            "train_vs_valid": leakage_metrics(train_records, valid_records),
        },
    }

    return metrics


def create_preprocessing_report(metrics: Dict[str, Any]) -> str:
    variant = metrics["variant"]
    counts = metrics["counts"]
    train_m = metrics["splits"]["train"]
    valid_m = metrics["splits"]["valid"]

    def pct(x):
        if x is None:
            return "n/a"
        return f"{x * 100:.2f}%"

    lines = []
    lines.append(f"# Preprocessing Report — {variant}")
    lines.append("")
    lines.append(f"Created at: `{metrics['created_at']}`")
    lines.append("")
    lines.append("## Counts")
    lines.append("")
    lines.append("| Split | Count |")
    lines.append("|---|---:|")
    lines.append(f"| raw_train | {counts['raw_train_count']} |")
    lines.append(f"| raw_valid | {counts['raw_valid_count']} |")
    lines.append(f"| train output | {counts['train_output_count']} |")
    lines.append(f"| valid output | {counts['valid_output_count']} |")
    lines.append("")
    lines.append("## Key Train Metrics")
    lines.append("")
    lines.append("| Metric | Value |")
    lines.append("|---|---:|")
    lines.append(f"| Answer extract success | {pct(train_m['answer_extract_success_rate'])} |")
    lines.append(f"| Anchor has rate | {pct(train_m['anchor_has_rate'])} |")
    lines.append(f"| Anchor last line rate | {pct(train_m['anchor_last_line_rate'])} |")
    lines.append(f"| Exact duplicate remaining | {train_m['exact_duplicate_remaining']} |")
    lines.append(f"| Conflict query remaining | {train_m['conflict_query_remaining']} |")
    lines.append(f"| Conflict row remaining | {train_m['conflict_row_remaining']} |")
    lines.append(f"| Duplicate anchor count | {train_m['duplicate_anchor_count']} |")
    lines.append(f"| Bad final answer candidate count | {train_m['bad_final_answer_candidate_count']} |")
    lines.append(f"| Pure decimal comma answer count | {train_m['pure_decimal_comma_answer_count']} |")
    lines.append(f"| LaTeX preservation ratio | {pct(train_m['latex_preservation_ratio'])} |")
    lines.append(f"| Asy after count | {train_m['asy_count_after']} |")
    lines.append("")
    lines.append("## Key Valid Metrics")
    lines.append("")
    lines.append("| Metric | Value |")
    lines.append("|---|---:|")
    lines.append(f"| Answer extract success | {pct(valid_m['answer_extract_success_rate'])} |")
    lines.append(f"| Anchor has rate | {pct(valid_m['anchor_has_rate'])} |")
    lines.append(f"| Anchor last line rate | {pct(valid_m['anchor_last_line_rate'])} |")
    lines.append(f"| Duplicate anchor count | {valid_m['duplicate_anchor_count']} |")
    lines.append(f"| Bad final answer candidate count | {valid_m['bad_final_answer_candidate_count']} |")
    lines.append(f"| Pure decimal comma answer count | {valid_m['pure_decimal_comma_answer_count']} |")
    lines.append("")
    lines.append("## Leakage Check")
    lines.append("")
    lines.append("| Pair | Group overlap | Query overlap | Query-response overlap |")
    lines.append("|---|---:|---:|---:|")

    for pair, values in metrics["leakage_check"].items():
        if values is None:
            continue
        lines.append(
            f"| {pair} | {values['group_overlap_count']} | {values['query_overlap_count']} | {values['query_response_overlap_count']} |"
        )

    lines.append("")
    lines.append("Note: external valid data may intentionally overlap with train depending on how the dataset was built. Treat leakage numbers as diagnostics, not always as hard failure.")
    lines.append("")
    lines.append("## Drop Summary")
    lines.append("")

    if metrics["drop_summary"]:
        lines.append("| Reason | Count |")
        lines.append("|---|---:|")
        for k, v in sorted(metrics["drop_summary"].items()):
            lines.append(f"| {k} | {v} |")
    else:
        lines.append("No dropped records.")

    lines.append("")

    return "\n".join(lines)

## 10. Run preprocessing and export

In [22]:
def prepare_output_dir() -> None:
    if OUTPUT_DIR.exists() and OVERWRITE_OUTPUT:
        shutil.rmtree(OUTPUT_DIR)

    ensure_dir(OUTPUT_DIR)
    ensure_dir(MANUAL_REVIEW_DIR)

prepare_output_dir()

all_transform_logs: List[Dict[str, Any]] = []
all_sample_diffs: List[Dict[str, Any]] = []
all_drop_logs: List[Dict[str, Any]] = []
split_outputs: Dict[str, List[Dict[str, Any]]] = {}

for split_name, bases in base_splits.items():
    processed = []

    for b in bases:
        record, tlogs, diffs = preprocess_one_record(
            base=b,
            config=V06_CONFIG,
            variant_name=VARIANT_NAME,
            split_name=split_name,
        )

        processed.append(record)
        all_transform_logs.extend(tlogs)

        remaining = MAX_SAMPLE_DIFFS - len(all_sample_diffs)
        if remaining > 0:
            all_sample_diffs.extend(diffs[:remaining])

    # Train only: dedup + conflict handling.
    # Valid should remain an external evaluation set.
    if split_name == "train":
        processed, drop_logs = apply_dedup_and_conflicts(
            records=processed,
            variant_name=VARIANT_NAME,
            config=V06_CONFIG,
        )
        all_drop_logs.extend(drop_logs)

    split_outputs[split_name] = processed

# Main outputs.
write_jsonl(split_outputs["train"], OUTPUT_DIR / "train.jsonl")
write_jsonl(split_outputs["valid"], OUTPUT_DIR / "valid.jsonl")

# Compatibility aliases for old fine-tuning notebook.
# These are the same as valid.jsonl.
write_jsonl(split_outputs["valid"], OUTPUT_DIR / "valid_grouped.jsonl")
write_jsonl(split_outputs["valid"], OUTPUT_DIR / "valid_public.jsonl")

# Logs.
write_jsonl(all_drop_logs, OUTPUT_DIR / "drop_log.jsonl")
write_jsonl(all_transform_logs, OUTPUT_DIR / "transform_log.jsonl")
write_jsonl(all_sample_diffs, OUTPUT_DIR / "sample_diffs.jsonl")

variant_config_out = {
    "variant": VARIANT_NAME,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "config": V06_CONFIG,
    "input_paths": {
        "raw_train": str(RAW_TRAIN_PATH),
        "raw_valid": str(RAW_VALID_PATH),
    },
    "output_dir": str(OUTPUT_DIR),
    "split_policy": {
        "use_external_valid": USE_EXTERNAL_VALID,
        "create_train_splits": CREATE_TRAIN_SPLITS,
        "train_split_note": "No train split was created. External valid file is used directly.",
        "seed": SEED,
    },
}
write_json(variant_config_out, OUTPUT_DIR / "variant_config.json")

write_json(split_payload, OUTPUT_DIR / "input_manifest.json")

metrics = build_metrics_pretrain(
    variant_name=VARIANT_NAME,
    config=V06_CONFIG,
    split_outputs=split_outputs,
    drop_logs=all_drop_logs,
    transform_logs=all_transform_logs,
)

write_json(metrics, OUTPUT_DIR / "metrics_pretrain.json")
write_text(create_preprocessing_report(metrics), OUTPUT_DIR / "preprocessing_report.md")

print(
    f"Done {VARIANT_NAME}: "
    f"train={len(split_outputs['train'])}, "
    f"valid={len(split_outputs['valid'])}, "
    f"drops={len(all_drop_logs)}"
)

print("Output:", OUTPUT_DIR)

Done v06_keep_asy_if_short: train=94992, valid=1000, drops=408
Output: D:\.Kỳ II năm Ba\DL\BTL\viet-gpt2-math-word-problems\data\processed\v06_keep_asy_if_short


## 11. Final checks

In [23]:
required_files = [
    OUTPUT_DIR / "train.jsonl",
    OUTPUT_DIR / "valid.jsonl",
    OUTPUT_DIR / "valid_grouped.jsonl",
    OUTPUT_DIR / "valid_public.jsonl",
    OUTPUT_DIR / "metrics_pretrain.json",
    OUTPUT_DIR / "variant_config.json",
    OUTPUT_DIR / "preprocessing_report.md",
    OUTPUT_DIR / "drop_log.jsonl",
    OUTPUT_DIR / "transform_log.jsonl",
    OUTPUT_DIR / "sample_diffs.jsonl",
    OUTPUT_DIR / "input_manifest.json",
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("Missing output files:\n" + "\n".join(missing))

train_metrics = metrics["splits"]["train"]
valid_metrics = metrics["splits"]["valid"]
train_vs_valid = metrics["leakage_check"]["train_vs_valid"]

hard_checks = {
    "train_anchor_has_rate_is_1": train_metrics["anchor_has_rate"] == 1.0,
    "train_anchor_last_line_rate_is_1": train_metrics["anchor_last_line_rate"] == 1.0,
    "train_exact_duplicate_remaining_is_0": train_metrics["exact_duplicate_remaining"] == 0,
    "train_conflict_query_remaining_is_0": train_metrics["conflict_query_remaining"] == 0,
    "train_conflict_row_remaining_is_0": train_metrics["conflict_row_remaining"] == 0,
    "train_bad_final_answer_candidate_is_0": train_metrics["bad_final_answer_candidate_count"] == 0,
    "train_pure_decimal_comma_answer_is_0": train_metrics["pure_decimal_comma_answer_count"] == 0,

    "valid_anchor_has_rate_is_1": valid_metrics["anchor_has_rate"] == 1.0,
    "valid_anchor_last_line_rate_is_1": valid_metrics["anchor_last_line_rate"] == 1.0,
    "valid_bad_final_answer_candidate_is_0": valid_metrics["bad_final_answer_candidate_count"] == 0,
    "valid_pure_decimal_comma_answer_is_0": valid_metrics["pure_decimal_comma_answer_count"] == 0,
}

diagnostic_checks = {
    "train_vs_valid_group_overlap_count": train_vs_valid["group_overlap_count"],
    "train_vs_valid_query_overlap_count": train_vs_valid["query_overlap_count"],
    "train_vs_valid_query_response_overlap_count": train_vs_valid["query_response_overlap_count"],
}

print("Hard checks:")
print(json.dumps(hard_checks, ensure_ascii=False, indent=2))

print("\nDiagnostic checks:")
print(json.dumps(diagnostic_checks, ensure_ascii=False, indent=2))

if not all(hard_checks.values()):
    raise AssertionError("Some final quality checks failed. Inspect metrics_pretrain.json and drop_log.jsonl.")

print("PASS: clean external-valid preprocessing output is ready for fine-tuning.")

Hard checks:
{
  "train_anchor_has_rate_is_1": true,
  "train_anchor_last_line_rate_is_1": true,
  "train_exact_duplicate_remaining_is_0": true,
  "train_conflict_query_remaining_is_0": true,
  "train_conflict_row_remaining_is_0": true,
  "train_bad_final_answer_candidate_is_0": true,
  "train_pure_decimal_comma_answer_is_0": true,
  "valid_anchor_has_rate_is_1": true,
  "valid_anchor_last_line_rate_is_1": true,
  "valid_bad_final_answer_candidate_is_0": true,
  "valid_pure_decimal_comma_answer_is_0": true
}

Diagnostic checks:
{
  "train_vs_valid_group_overlap_count": 909,
  "train_vs_valid_query_overlap_count": 18,
  "train_vs_valid_query_response_overlap_count": 0
}
PASS: clean external-valid preprocessing output is ready for fine-tuning.


## 12. Next step

Dùng các file sau cho notebook fine-tuning:

```python
TRAIN_FILE = DATA_DIR / 'processed' / 'basic_preprocessing' / 'train.jsonl'
VALID_FILE = DATA_DIR / 'processed' / 'basic_preprocessing' / 'valid_grouped.jsonl'
```

Khi fine-tune, thêm EOS cuối response và generate với `eos_token_id` để model học dừng đúng điểm.
